# ETF Valuation & Momentum Monitor
- 차트 1: 멀티플 Z-Score vs 모멘텀 스코어
- 차트 2: 멀티플 Z-Score vs 자금유입강도

각 ETF의 현재 위치와 1주일 전 위치를 화살표로 연결하여 표시

## 1. 파일 업로드

In [ ]:
from google.colab import files
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import requests
import io
import warnings
warnings.filterwarnings('ignore')

# 한글 폰트 설정 (Colab)
!apt-get install -y fonts-nanum > /dev/null 2>&1
fontpath = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'
font = fm.FontProperties(fname=fontpath, size=10)
plt.rc('font', family='NanumGothic')
fm._rebuild()
plt.rcParams['axes.unicode_minus'] = False
print("한글 폰트 설정 완료")

print("\n필요한 파일 7개를 업로드해주세요:")
print("1. 1_pe_now.txt - 현재 기준 5년치 PER")
print("2. 2_pe_1w.txt - 1주일 전 기준 5년치 PER")
print("3. 3_return.txt - 모멘텀 스코어")
print("4. 4_flow.txt - 자금유입 데이터")
print("5. 5_pbr_now.txt - 현재 기준 5년치 PBR")
print("6. 6_pbr_1w.txt - 1주일 전 기준 5년치 PBR")
print("7. 7_theme.txt - 테마 정의")
print()

uploaded = files.upload()

## 2. 설정

In [ ]:
# PBR로 처리해야 하는 ETF 리스트 (추후 추가 가능)
PBR_ETF_LIST = ['BLOK-US', 'SOXX-US']

# 텔레그램 설정
BOT_TOKEN = "8328122559:AAEXkzJnxtljMON_Obt4uyb5PzJvx5-IS64"
CHAT_ID = "7481149685"

import os
os.environ['TELEGRAM_BOT_TOKEN'] = BOT_TOKEN
os.environ['TELEGRAM_CHAT_ID'] = CHAT_ID

print(f"PBR 기준 ETF: {PBR_ETF_LIST}")
print("텔레그램 설정 완료")

## 3. 데이터 로드 및 전처리

In [ ]:
def load_pe_data(filename, uploaded_files):
    """PER/PBR 데이터 로드 (컬럼=ETF, 행=날짜)"""
    print(f"  로딩: {filename}...")
    content = uploaded_files[filename].decode('utf-8')
    df = pd.read_csv(io.StringIO(content), sep='\t', index_col=0)
    df.index = pd.to_datetime(df.index, format='%m/%d/%Y')
    df = df.replace('#N/A', np.nan).replace('', np.nan)
    df = df.astype(float)
    print(f"    완료: {df.shape[0]}행 x {df.shape[1]}컬럼")
    return df

def load_return_data(filename, uploaded_files):
    """모멘텀 스코어 데이터 로드 (행=ETF, 컬럼=지표)"""
    print(f"  로딩: {filename}...")
    content = uploaded_files[filename].decode('utf-8')
    df = pd.read_csv(io.StringIO(content), sep='\t', index_col=0)
    df = df.replace('#N/A', np.nan).replace('', np.nan)
    for col in ['Score', 'Score_1W']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    print(f"    완료: {df.shape[0]}행 x {df.shape[1]}컬럼")
    return df

def load_flow_data(filename, uploaded_files):
    """자금유입 데이터 로드 (행=ETF, 컬럼=지표)"""
    print(f"  로딩: {filename}...")
    content = uploaded_files[filename].decode('utf-8')
    df = pd.read_csv(io.StringIO(content), sep='\t', index_col=0, skiprows=[1])
    df = df.replace('#N/A', np.nan).replace('', np.nan)
    # 모든 숫자 컬럼 변환
    for col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    print(f"    완료: {df.shape[0]}행 x {df.shape[1]}컬럼")
    return df

def load_theme_data(filename, uploaded_files):
    """테마 정의 로드 (테마명 -> ETF 리스트)"""
    print(f"  로딩: {filename}...")
    content = uploaded_files[filename].decode('utf-8')
    lines = content.strip().split('\n')
    
    themes = {}
    for line in lines:
        parts = [p.strip() for p in line.split('\t')]
        theme_name = parts[0]
        # ETF 이름에 -US 붙이기 (빈값 제외)
        etfs = [f"{p}-US" for p in parts[1:] if p]
        themes[theme_name] = etfs
    
    print(f"    완료: {len(themes)}개 테마")
    for theme, etfs in themes.items():
        print(f"      {theme}: {len(etfs)}개 ETF")
    return themes

# 데이터 로드
print("데이터 로드 중...")
pe_now = load_pe_data('1_pe_now.txt', uploaded)
pe_1w = load_pe_data('2_pe_1w.txt', uploaded)
return_df = load_return_data('3_return.txt', uploaded)
flow_df = load_flow_data('4_flow.txt', uploaded)
pbr_now = load_pe_data('5_pbr_now.txt', uploaded)
pbr_1w = load_pe_data('6_pbr_1w.txt', uploaded)
themes = load_theme_data('7_theme.txt', uploaded)

print("\n모든 파일 로드 완료!")

## 4. Z-Score 계산

In [ ]:
def calculate_zscore(df, exclude_latest=True):
    """
    5년치 데이터로 Z-Score 계산
    - exclude_latest=True: 최신값 제외하고 평균/표준편차 계산 (권장)
    - Z = (현재값 - 과거평균) / 과거표준편차
    """
    result = {}
    
    for etf in df.columns:
        series = df[etf].dropna()
        
        if len(series) < 10:  # 데이터가 너무 적으면 스킵
            result[etf] = np.nan
            continue
        
        # 날짜 기준 정렬 (최신이 맨 앞)
        series = series.sort_index(ascending=False)
        
        current_value = series.iloc[0]  # 최신값
        
        if exclude_latest:
            # 최신값 제외하고 평균/표준편차 계산
            historical = series.iloc[1:]
        else:
            historical = series
        
        mean = historical.mean()
        std = historical.std()
        
        if std == 0 or np.isnan(std):
            result[etf] = np.nan
        else:
            zscore = (current_value - mean) / std
            result[etf] = zscore
    
    return result

# PER Z-Score 계산 (현재 & 1주일 전)
print("Z-Score 계산 중...")

# 현재 기준 Z-Score
zscore_now = calculate_zscore(pe_now, exclude_latest=True)
zscore_1w = calculate_zscore(pe_1w, exclude_latest=True)

# PBR 기준 ETF는 PBR Z-Score로 대체
pbr_zscore_now = calculate_zscore(pbr_now, exclude_latest=True)
pbr_zscore_1w = calculate_zscore(pbr_1w, exclude_latest=True)

for etf in PBR_ETF_LIST:
    if etf in pbr_zscore_now:
        zscore_now[etf] = pbr_zscore_now[etf]
        print(f"{etf}: PBR Z-Score 사용 (Now: {pbr_zscore_now[etf]:.2f})")
    if etf in pbr_zscore_1w:
        zscore_1w[etf] = pbr_zscore_1w[etf]
        print(f"{etf}: PBR Z-Score 사용 (1W: {pbr_zscore_1w[etf]:.2f})")

print(f"\nZ-Score 계산 완료: {len([v for v in zscore_now.values() if not np.isnan(v)])}개 ETF")

## 5. 최종 데이터 통합

In [ ]:
# 테마별 데이터 집계
print("테마별 데이터 집계 중...")

theme_data = []

for theme_name, etf_list in themes.items():
    # 해당 테마의 ETF들만 필터링
    valid_etfs_zscore = [e for e in etf_list if e in zscore_now and not np.isnan(zscore_now.get(e, np.nan))]
    valid_etfs_momentum = [e for e in etf_list if e in return_df.index]
    valid_etfs_flow = [e for e in etf_list if e in flow_df.index]
    
    # Z-Score 평균 (현재 & 1주전)
    if valid_etfs_zscore:
        zscore_now_avg = np.nanmean([zscore_now[e] for e in valid_etfs_zscore])
        zscore_1w_avg = np.nanmean([zscore_1w.get(e, np.nan) for e in valid_etfs_zscore])
    else:
        zscore_now_avg = np.nan
        zscore_1w_avg = np.nan
    
    # 모멘텀 스코어 평균 (현재 & 1주전)
    if valid_etfs_momentum:
        momentum_now_avg = return_df.loc[valid_etfs_momentum, 'Score'].mean()
        momentum_1w_avg = return_df.loc[valid_etfs_momentum, 'Score_1W'].mean()
    else:
        momentum_now_avg = np.nan
        momentum_1w_avg = np.nan
    
    # Flow 계산: sum(WF-0W) / sum(AUM-1W) * 100 (%)
    if valid_etfs_flow:
        # 현재: WF-0W / AUM-1W
        wf_0w_sum = flow_df.loc[valid_etfs_flow, 'WF-0W'].sum()
        aum_1w_sum = flow_df.loc[valid_etfs_flow, 'AUM-1W'].sum()
        flow_now = (wf_0w_sum / aum_1w_sum * 100) if aum_1w_sum != 0 else np.nan
        
        # 1주전: WF-1W / AUM-2W
        wf_1w_sum = flow_df.loc[valid_etfs_flow, 'WF-1W'].sum()
        aum_2w_sum = flow_df.loc[valid_etfs_flow, 'AUM-2W'].sum()
        flow_1w = (wf_1w_sum / aum_2w_sum * 100) if aum_2w_sum != 0 else np.nan
    else:
        flow_now = np.nan
        flow_1w = np.nan
    
    theme_data.append({
        'Theme': theme_name,
        'ZScore_Now': zscore_now_avg,
        'ZScore_1W': zscore_1w_avg,
        'Momentum_Now': momentum_now_avg,
        'Momentum_1W': momentum_1w_avg,
        'Flow_Now': flow_now,
        'Flow_1W': flow_1w,
        'ETF_Count': len(etf_list)
    })

df_theme = pd.DataFrame(theme_data)
df_theme = df_theme.set_index('Theme')

# 유효한 데이터만 필터링 (차트별로)
df_chart1 = df_theme.dropna(subset=['ZScore_Now', 'ZScore_1W', 'Momentum_Now', 'Momentum_1W'])
df_chart2 = df_theme.dropna(subset=['ZScore_Now', 'ZScore_1W', 'Flow_Now', 'Flow_1W'])

print(f"\n차트1 (멀티플 vs 모멘텀): {len(df_chart1)}개 테마")
print(f"차트2 (멀티플 vs 플로우): {len(df_chart2)}개 테마")

# 데이터 미리보기
print("\n=== 테마별 데이터 ===")
print(df_theme.round(3))

## 6. 차트 생성

In [ ]:
def create_scatter_chart(df, x_col_now, x_col_1w, y_col_now, y_col_1w, 
                         title, xlabel, ylabel, filename, x_is_percent=False):
    """
    스캐터 차트 생성
    - 각 테마마다 현재(점) + 1주전(점) + 화살표 연결
    - 대각선 기준선 (좌하단 -> 우상단)
    """
    fig, ax = plt.subplots(figsize=(14, 10))
    
    # 색상 팔레트
    colors = plt.cm.tab20(np.linspace(0, 1, len(df)))
    
    # 데이터 범위 계산 (기준선용)
    x_all = pd.concat([df[x_col_now], df[x_col_1w]])
    y_all = pd.concat([df[y_col_now], df[y_col_1w]])
    
    x_min, x_max = x_all.min(), x_all.max()
    y_min, y_max = y_all.min(), y_all.max()
    
    # 마진 추가
    x_margin = (x_max - x_min) * 0.15
    y_margin = (y_max - y_min) * 0.15
    
    # 대각선 기준선 (좌하단 -> 우상단)
    x_range = x_max - x_min
    y_range = y_max - y_min
    
    line_x = np.array([x_min - x_margin, x_max + x_margin])
    line_y = y_min + (line_x - x_min) * y_range / x_range
    
    ax.plot(line_x, line_y, 'k--', alpha=0.5, linewidth=1.5, label='기준선')
    
    # 각 테마 플롯
    for i, (theme, row) in enumerate(df.iterrows()):
        x_now = row[x_col_now]
        x_1w = row[x_col_1w]
        y_now = row[y_col_now]
        y_1w = row[y_col_1w]
        
        color = colors[i]
        
        # 1주전 위치 (작은 점, 연한 색)
        ax.scatter(x_1w, y_1w, c=[color], s=50, alpha=0.4, marker='o')
        
        # 현재 위치 (큰 점)
        ax.scatter(x_now, y_now, c=[color], s=120, alpha=0.9, marker='o', edgecolors='black', linewidths=0.5)
        
        # 화살표 (1주전 -> 현재)
        ax.annotate('', xy=(x_now, y_now), xytext=(x_1w, y_1w),
                    arrowprops=dict(arrowstyle='->', color=color, alpha=0.6, lw=1.5))
        
        # 라벨 (현재 위치에 테마명 표시)
        ax.annotate(theme, (x_now, y_now), fontsize=9, alpha=0.9, fontweight='bold',
                    xytext=(8, 8), textcoords='offset points')
    
    # 축 설정
    ax.set_xlim(x_min - x_margin, x_max + x_margin)
    ax.set_ylim(y_min - y_margin, y_max + y_margin)
    ax.set_xlabel(xlabel, fontsize=12)
    ax.set_ylabel(ylabel, fontsize=12)
    ax.set_title(title, fontsize=14, fontweight='bold')
    
    # X축 % 포맷
    if x_is_percent:
        from matplotlib.ticker import FuncFormatter
        ax.xaxis.set_major_formatter(FuncFormatter(lambda x, p: f'{x:.2f}%'))
    
    # 그리드
    ax.grid(True, alpha=0.3)
    ax.axhline(y=0, color='gray', linewidth=0.8, alpha=0.5)
    ax.axvline(x=0, color='gray', linewidth=0.8, alpha=0.5)
    
    # 범례 (영역 설명)
    ax.text(0.02, 0.98, '비중 축소 고려\n(높은 밸류에이션, 낮은 모멘텀/플로우)', 
            transform=ax.transAxes, fontsize=10, verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor='lightcoral', alpha=0.3))
    ax.text(0.98, 0.02, '비중 확대 고려\n(낮은 밸류에이션, 높은 모멘텀/플로우)', 
            transform=ax.transAxes, fontsize=10, verticalalignment='bottom', horizontalalignment='right',
            bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.3))
    
    plt.tight_layout()
    
    # 파일 저장
    plt.savefig(filename, dpi=150, bbox_inches='tight', facecolor='white')
    print(f"차트 저장: {filename}")
    
    return fig

# 차트 1: 멀티플 Z-Score vs 모멘텀 스코어
fig1 = create_scatter_chart(
    df_chart1,
    x_col_now='Momentum_Now', x_col_1w='Momentum_1W',
    y_col_now='ZScore_Now', y_col_1w='ZScore_1W',
    title='테마별 밸류에이션 vs 모멘텀 (주간 변화)',
    xlabel='모멘텀 스코어',
    ylabel='밸류에이션 Z-Score (5Y PER/PBR)',
    filename='chart1_valuation_momentum.png'
)
plt.show()

# 차트 2: 멀티플 Z-Score vs 자금유입강도
fig2 = create_scatter_chart(
    df_chart2,
    x_col_now='Flow_Now', x_col_1w='Flow_1W',
    y_col_now='ZScore_Now', y_col_1w='ZScore_1W',
    title='테마별 밸류에이션 vs 자금유입강도 (주간 변화)',
    xlabel='자금유입강도 (%)',
    ylabel='밸류에이션 Z-Score (5Y PER/PBR)',
    filename='chart2_valuation_flow.png',
    x_is_percent=True
)
plt.show()

## 7. 텔레그램 전송

In [ ]:
class TelegramSender:
    def __init__(self, bot_token, chat_id):
        self.bot_token = bot_token
        self.chat_id = chat_id
        self.base_url = f"https://api.telegram.org/bot{bot_token}"
    
    def send_photo(self, photo_path, caption=None):
        """이미지 파일 전송"""
        url = f"{self.base_url}/sendPhoto"
        
        with open(photo_path, 'rb') as photo:
            files = {'photo': photo}
            data = {'chat_id': self.chat_id}
            if caption:
                data['caption'] = caption
            
            response = requests.post(url, files=files, data=data)
        
        if response.status_code == 200:
            return True
        else:
            print(f"Error: {response.text}")
            return False

# 텔레그램 전송 실행
if BOT_TOKEN and CHAT_ID:
    print("텔레그램 전송 중...")
    sender = TelegramSender(BOT_TOKEN, CHAT_ID)
    
    from datetime import datetime
    date_str = datetime.now().strftime('%Y-%m-%d')
    
    # 차트 1 전송
    if sender.send_photo('chart1_valuation_momentum.png', 
                         caption=f"테마별 밸류에이션 vs 모멘텀 ({date_str})\n"
                                 f"- 좌상단: 비중 축소 고려 (고평가 + 약모멘텀)\n"
                                 f"- 우하단: 비중 확대 고려 (저평가 + 강모멘텀)"):
        print("차트1 전송 완료")
    else:
        print("차트1 전송 실패")
    
    # 차트 2 전송
    if sender.send_photo('chart2_valuation_flow.png',
                         caption=f"테마별 밸류에이션 vs 자금유입강도 ({date_str})\n"
                                 f"- 좌상단: 비중 축소 고려 (고평가 + 자금유출)\n"
                                 f"- 우하단: 비중 확대 고려 (저평가 + 자금유입)"):
        print("차트2 전송 완료")
    else:
        print("차트2 전송 실패")
    
    print("\n텔레그램 전송 완료!")
else:
    print("텔레그램 설정이 필요합니다.")

## 8. 데이터 확인 (선택사항)

In [ ]:
# 전체 테마 데이터 확인
print("=== 테마별 Z-Score 및 지표 ===")
display_df = df_theme.copy()
display_df = display_df.round(3)
display_df = display_df.sort_values('ZScore_Now', ascending=False)
display(display_df)

In [ ]:
# 주요 테마 위치 변화 요약
print("=== 주간 변화 요약 ===")
df_summary = df_theme.copy()
df_summary['ZScore_Change'] = df_summary['ZScore_Now'] - df_summary['ZScore_1W']
df_summary['Momentum_Change'] = df_summary['Momentum_Now'] - df_summary['Momentum_1W']
df_summary['Flow_Change'] = df_summary['Flow_Now'] - df_summary['Flow_1W']

# Z-Score 상승 Top 5
print("\n[Z-Score 상승 Top 5 (밸류에이션 부담 증가)]")
print(df_summary.nlargest(5, 'ZScore_Change')[['ZScore_Now', 'ZScore_1W', 'ZScore_Change']].round(3))

# Z-Score 하락 Top 5
print("\n[Z-Score 하락 Top 5 (밸류에이션 매력 증가)]")
print(df_summary.nsmallest(5, 'ZScore_Change')[['ZScore_Now', 'ZScore_1W', 'ZScore_Change']].round(3))

# 모멘텀 상승 Top 5
print("\n[모멘텀 상승 Top 5]")
print(df_summary.nlargest(5, 'Momentum_Change')[['Momentum_Now', 'Momentum_1W', 'Momentum_Change']].round(3))

# 자금유입 상승 Top 5
print("\n[자금유입 상승 Top 5]")
print(df_summary.nlargest(5, 'Flow_Change')[['Flow_Now', 'Flow_1W', 'Flow_Change']].round(3))